In [ ]:
import os, math, time, warnings
warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
CONFIG = {
    "CSV_PATH": "/content/train.csv",

    # Splits
    "TEST_SIZE": 0.25,
    "VAL_SIZE": 0.20,

    # Feature controls
    "TOPK_ONEHOT": 100,     # cap per-column one-hot (others -> "__RARE__")

    "CAT2VEC_EPOCHS": 25,
    "CAT2VEC_MIN_COUNT": 3,
    "CAT2VEC_MAX_EMB_DIM": 24,
    "EMBED_SAMPLE": None,     # sample for embeddings on large data; set None for full train
    "BATCH_SIZE": 1024,
    "LEARNING_RATE": 1e-3,
    "POS_ENCODING_DIM": 12,

    # Models to include
    "INCLUDE_LOGREG": True,
    "INCLUDE_RF": True,
    "INCLUDE_XGB": True,
    "INCLUDE_SVM": True,  
    "INCLUDE_KNN": True,

    # Model sizes
    "RF_TREES": 300,
    "XGB_TREES": 600,

    # Output
    "RESULTS_CSV": "catindat2_results_quality.csv",
}

import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple

# sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    precision_recall_fscore_support, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# xgboost / tensorflow
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping as XgbEarlyStopping
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks

np.random.seed(42)
tf.random.set_seed(42)

# Optional sparse handling
try:
    import scipy.sparse as sp
except Exception:
    sp = None

# ---------------------------
# Utilities: slice sanitizers
# ---------------------------
def _to_str_np(X):
    X = pd.DataFrame(X).copy()
    for c in X.columns: X[c] = X[c].astype(str)
    return X.to_numpy()

def _to_num_np(X):
    X = pd.DataFrame(X).copy()
    for c in X.columns: X[c] = pd.to_numeric(X[c], errors="coerce")
    return X.fillna(0.0).to_numpy()

stringify = FunctionTransformer(_to_str_np, feature_names_out="one-to-one")
numify    = FunctionTransformer(_to_num_np, feature_names_out="one-to-one")

# ---------------------------
# TopK reducer (limit OHE width)
# ---------------------------
class TopKReducer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=100, other_token="__RARE__"):
        self.top_k = top_k
        self.other_token = other_token

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.kept_ = {}
        for c in self.columns_:
            top = X[c].astype(str).value_counts(dropna=False).head(self.top_k).index.tolist()
            self.kept_[c] = set(top)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.columns_:
            mask = ~X[c].astype(str).isin(self.kept_[c])
            X.loc[mask, c] = self.other_token
            X[c] = X[c].astype(str)
        return X

# ---------------------------
# Positional Encoding (sin/cos)
# ---------------------------
def positional_encoding_1d(length: int, d_model: int, n: float = 10000.0) -> np.ndarray:
    positions = np.arange(length)[:, None]
    dims = np.arange(d_model)[None, :]
    div_term = np.power(n, (2 * (dims // 2)) / d_model)
    sin_vals = np.sin(positions / div_term)
    cos_vals = np.cos(positions / div_term)
    pe = np.zeros((length, d_model), dtype=np.float32)
    pe[:, 0::2] = sin_vals[:, 0::2]
    pe[:, 1::2] = cos_vals[:, 1::2]
    return pe.astype(np.float32)

# ---------------------------
# Cat2Vec (nominal) – QUALITY
# ---------------------------
class Cat2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, epochs=25, batch_size=1024, lr=1e-3, min_count=3, max_emb_dim=24, sample_n=150_000):
        self.epochs = epochs; self.batch_size = batch_size; self.lr = lr
        self.min_count = min_count; self.max_emb_dim = max_emb_dim; self.sample_n = sample_n

    def _fit_single(self, x_col: pd.Series, y: np.ndarray):
        vc = x_col.value_counts()
        kept = vc[vc >= self.min_count].index.tolist()
        cat2idx = {cat: i+1 for i, cat in enumerate(sorted(map(str, kept)))}  # 0 OOV
        K = len(cat2idx) + 1
        emb_dim = int(max(2, math.sqrt(K))) + 1
        emb_dim = min(self.max_emb_dim, emb_dim)

        idxs = x_col.astype(str).map(cat2idx).fillna(0).astype(int).values
        yy = y.copy()

        # optional sampling for speed
        if (self.sample_n is not None) and (len(idxs) > self.sample_n):
            rs = np.random.RandomState(42)
            sel = rs.choice(len(idxs), self.sample_n, replace=False)
            idxs = idxs[sel]; yy = yy[sel]

        num_classes = int(np.max(yy)) + 1
        inp = layers.Input(shape=(1,), dtype="int32")
        emb = layers.Embedding(K, emb_dim, name="emb")(inp)
        x = layers.Flatten()(emb)
        x = layers.Dense(32, activation="relu")(x)
        x = layers.Dense(16, activation="relu")(x)
        out = layers.Dense(num_classes, activation="softmax")(x)
        model = models.Model(inp, out)
        model.compile(optimizers.Adam(self.lr), losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        es = callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True, verbose=0)
        model.fit(idxs, yy, validation_split=0.1, epochs=self.epochs, batch_size=self.batch_size, verbose=0, callbacks=[es])
        return cat2idx, model.get_layer("emb").get_weights()[0]

    def fit(self, X, y):
        X = pd.DataFrame(X); self.cols_ = list(X.columns)
        if getattr(y, "dtype", None) is not None and y.dtype.kind not in "iu":
            _, y = np.unique(y, return_inverse=True)
        self.maps_, self.embs_, self.total_dim_ = {}, {}, 0
        for col in self.cols_:
            m, e = self._fit_single(X[col], y)
            self.maps_[col], self.embs_[col] = m, e
            self.total_dim_ += e.shape[1]
        return self

    def transform(self, X):
        X = pd.DataFrame(X); outs = []
        for col in self.cols_:
            m, e = self.maps_[col], self.embs_[col]
            idxs = X[col].astype(str).map(m).fillna(0).astype(int).values
            outs.append(e[idxs])
        return (np.concatenate(outs, 1).astype(np.float32)
                if outs else np.zeros((len(X), 0), np.float32))

# ---------------------------
# Cat2Vec_wpos (ordinal + PE) – QUALITY
# ---------------------------
class Cat2VecWPosTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, ordinal_order_map: Optional[Dict[str, List]] = None,
                 pos_dim=12, epochs=25, batch_size=1024, lr=1e-3,
                 min_count=3, max_emb_dim=24, sample_n=150_000):
        self.ordinal_order_map = ordinal_order_map or {}
        self.pos_dim = pos_dim; self.epochs = epochs; self.batch_size = batch_size; self.lr = lr
        self.min_count = min_count; self.max_emb_dim = max_emb_dim; self.sample_n = sample_n

    def _rank_map(self, x_col: pd.Series, col: str):
        if self.ordinal_order_map.get(col):
            ordered = list(map(str, self.ordinal_order_map[col]))
        else:
            ordered = sorted(map(str, x_col.dropna().unique()))
        return {cat:i for i,cat in enumerate(ordered)}

    def _fit_single(self, x_col: pd.Series, y: np.ndarray, rank_map: Dict[str,int]):
        vc = x_col.value_counts()
        kept = vc[vc >= self.min_count].index.tolist()
        cat2idx = {cat: i+1 for i, cat in enumerate(sorted(map(str, kept)))}  # 0 OOV
        K = len(cat2idx) + 1
        emb_dim = int(max(2, math.sqrt(K))) + 1
        emb_dim = min(self.max_emb_dim, emb_dim)

        idxs = x_col.astype(str).map(cat2idx).fillna(0).astype(int).values
        yy = y.copy()
        if (self.sample_n is not None) and (len(idxs) > self.sample_n):
            rs = np.random.RandomState(42)
            sel = rs.choice(len(idxs), self.sample_n, replace=False)
            idxs = idxs[sel]; yy = yy[sel]

        num_classes = int(np.max(yy)) + 1
        inp = layers.Input(shape=(1,), dtype="int32")
        emb = layers.Embedding(K, emb_dim, name="emb")(inp)
        x = layers.Flatten()(emb)
        x = layers.Dense(32, activation="relu")(x)
        x = layers.Dense(16, activation="relu")(x)
        out = layers.Dense(num_classes, activation="softmax")(x)
        model = models.Model(inp, out)
        model.compile(optimizers.Adam(self.lr), losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        es = callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True, verbose=0)
        model.fit(idxs, yy, validation_split=0.1, epochs=self.epochs, batch_size=self.batch_size, verbose=0, callbacks=[es])
        return cat2idx, model.get_layer("emb").get_weights()[0]

    def fit(self, X, y):
        X = pd.DataFrame(X); self.cols_ = list(X.columns)
        if getattr(y, "dtype", None) is not None and y.dtype.kind not in "iu":
            _, y = np.unique(y, return_inverse=True)
        self.maps_, self.ranks_, self.embs_, self.total_dim_ = {}, {}, {}, 0
        for col in self.cols_:
            rmap = self._rank_map(X[col], col)
            m, e = self._fit_single(X[col], y, rmap)
            self.maps_[col], self.ranks_[col], self.embs_[col] = m, rmap, e
            self.total_dim_ += e.shape[1] + CONFIG["POS_ENCODING_DIM"]
        return self

    def transform(self, X):
        X = pd.DataFrame(X); outs=[]
        for col in self.cols_:
            m, rmap, e = self.maps_[col], self.ranks_[col], self.embs_[col]
            idxs  = X[col].astype(str).map(m).fillna(0).astype(int).values
            base  = e[idxs]
            ranks = X[col].astype(str).map(rmap).fillna(0).astype(int).values
            K = (max(rmap.values())+1) if rmap else 1
            pe_mat = positional_encoding_1d(K, CONFIG["POS_ENCODING_DIM"])
            pe = pe_mat[np.clip(ranks, 0, K-1)]
            outs.append(np.concatenate([base, pe], 1))
        return (np.concatenate(outs, 1).astype(np.float32)
                if outs else np.zeros((len(X), 0), np.float32))

# ---------------------------
# Build preprocessors (P1..P4)
# ---------------------------
def build_preprocessors(NOMINAL_COLS, ORDINAL_COLS, CONTINUOUS_COLS, ORDINAL_ORDER_MAP):
    topk = TopKReducer(top_k=CONFIG["TOPK_ONEHOT"])

    pre_1 = ColumnTransformer([
        ("ord", Pipeline([("to_str", stringify),
                          ("ordenc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                          ("sc", StandardScaler())]), ORDINAL_COLS),
        ("nom", Pipeline([("to_str", stringify),
                          ("topk", topk),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=True))]), NOMINAL_COLS),
        ("cont", Pipeline([("to_num", numify), ("sc", StandardScaler())]), CONTINUOUS_COLS),
    ], remainder="drop")

    pre_2 = ColumnTransformer([
        ("ord", Pipeline([("to_str", stringify),
                          ("ordenc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                          ("sc", StandardScaler())]), ORDINAL_COLS),
        ("nom", Pipeline([("to_str", stringify),
                          ("c2v", Cat2VecTransformer(
                              epochs=CONFIG["CAT2VEC_EPOCHS"], batch_size=CONFIG["BATCH_SIZE"],
                              lr=CONFIG["LEARNING_RATE"], min_count=CONFIG["CAT2VEC_MIN_COUNT"],
                              max_emb_dim=CONFIG["CAT2VEC_MAX_EMB_DIM"], sample_n=CONFIG["EMBED_SAMPLE"]
                          )),
                          ("sc", StandardScaler(with_mean=True))]), NOMINAL_COLS),
        ("cont", Pipeline([("to_num", numify), ("sc", StandardScaler())]), CONTINUOUS_COLS),
    ], remainder="drop")

    pre_3 = ColumnTransformer([
        ("ord", Pipeline([("to_str", stringify),
                          ("c2vw", Cat2VecWPosTransformer(
                              ordinal_order_map=ORDINAL_ORDER_MAP, pos_dim=CONFIG["POS_ENCODING_DIM"],
                              epochs=CONFIG["CAT2VEC_EPOCHS"], batch_size=CONFIG["BATCH_SIZE"],
                              lr=CONFIG["LEARNING_RATE"], min_count=CONFIG["CAT2VEC_MIN_COUNT"],
                              max_emb_dim=CONFIG["CAT2VEC_MAX_EMB_DIM"], sample_n=CONFIG["EMBED_SAMPLE"]
                          )),
                          ("sc", StandardScaler(with_mean=True))]), ORDINAL_COLS),
        ("nom", Pipeline([("to_str", stringify),
                          ("topk", topk),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=True))]), NOMINAL_COLS),
        ("cont", Pipeline([("to_num", numify), ("sc", StandardScaler())]), CONTINUOUS_COLS),
    ], remainder="drop")

    pre_4 = ColumnTransformer([
        ("ord", Pipeline([("to_str", stringify),
                          ("c2vw", Cat2VecWPosTransformer(
                              ordinal_order_map=ORDINAL_ORDER_MAP, pos_dim=CONFIG["POS_ENCODING_DIM"],
                              epochs=CONFIG["CAT2VEC_EPOCHS"], batch_size=CONFIG["BATCH_SIZE"],
                              lr=CONFIG["LEARNING_RATE"], min_count=CONFIG["CAT2VEC_MIN_COUNT"],
                              max_emb_dim=CONFIG["CAT2VEC_MAX_EMB_DIM"], sample_n=CONFIG["EMBED_SAMPLE"]
                          )),
                          ("sc", StandardScaler(with_mean=True))]), ORDINAL_COLS),
        ("nom", Pipeline([("to_str", stringify),
                          ("c2v", Cat2VecTransformer(
                              epochs=CONFIG["CAT2VEC_EPOCHS"], batch_size=CONFIG["BATCH_SIZE"],
                              lr=CONFIG["LEARNING_RATE"], min_count=CONFIG["CAT2VEC_MIN_COUNT"],
                              max_emb_dim=CONFIG["CAT2VEC_MAX_EMB_DIM"], sample_n=CONFIG["EMBED_SAMPLE"]
                          )),
                          ("sc", StandardScaler(with_mean=True))]), NOMINAL_COLS),
        ("cont", Pipeline([("to_num", numify), ("sc", StandardScaler())]), CONTINUOUS_COLS),
    ], remainder="drop")

    return [pre_1, pre_2, pre_3, pre_4]

# ---------------------------
# Classifiers
# ---------------------------
def get_classifiers(y_train: np.ndarray):
    pos = int((y_train == 1).sum()); neg = int((y_train == 0).sum())
    spw = max(1.0, neg / max(1, pos))  # for XGB imbalance

    clfs = {}
    if CONFIG["INCLUDE_LOGREG"]:
        clfs["LogReg"] = LogisticRegression(max_iter=400, class_weight="balanced")
    if CONFIG["INCLUDE_RF"]:
        clfs["RF"] = RandomForestClassifier(
            n_estimators=CONFIG["RF_TREES"], random_state=42, n_jobs=-1,
            class_weight="balanced_subsample"
        )
    if CONFIG["INCLUDE_XGB"]:
        clfs["XGB"] = XGBClassifier(
            n_estimators=CONFIG["XGB_TREES"], max_depth=6, learning_rate=0.07,
            subsample=0.9, colsample_bytree=0.9, tree_method="hist",
            eval_metric="logloss", random_state=42, scale_pos_weight=spw,
            n_jobs=-1
        )
    if CONFIG["INCLUDE_SVM"]:
        clfs["SVM"] = SVC(probability=True, kernel="rbf", C=2.0, gamma="scale",
                           random_state=42, class_weight="balanced")
    if CONFIG["INCLUDE_KNN"]:
        clfs["KNN"] = KNeighborsClassifier(n_neighbors=15)

    return clfs

# ---------------------------
# Threshold tuning (macro-F1) on validation scores
# ---------------------------
def tune_threshold(scores, y_true, metric="macro_f1"):
    ts = np.linspace(0.05, 0.95, 181)
    if metric == "macro_f1":
        vals = [f1_score(y_true, (scores>=t).astype(int), average="macro") for t in ts]
    else:
        vals = [f1_score(y_true, (scores>=t).astype(int), pos_label=1) for t in ts]
    idx = int(np.argmax(vals))
    return float(ts[idx]), float(vals[idx])

# ---------------------------
# Data loader with TRUE ordinal selection
# ---------------------------
def load_catindat2(csv_path: str):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)

    if "id" in df.columns: df = df.drop(columns=["id"])
    if "target" not in df.columns:
        raise ValueError("Expected 'target' column in the CSV.")
    df = df.rename(columns={"target": "y"})

    for c in df.select_dtypes(include=["object"]).columns:
        df[c] = df[c].astype(str).str.strip().replace({"nan":"unknown"})

    # Start with prefixes
    nominal = [c for c in df.columns if c.startswith("bin_") or c.startswith("nom_")]
    candidate_ords = [c for c in df.columns if c.startswith("ord_")]
    for extra in ["day", "month"]:
        if extra in df.columns: candidate_ords.append(extra)

    # TRUE ordinals only
    true_ords = []
    if "ord_0" in df.columns and (df["ord_0"].dtype != object):
        true_ords.append("ord_0")
    if "ord_1" in df.columns: true_ords.append("ord_1")
    if "ord_2" in df.columns: true_ords.append("ord_2")
    if "day"   in df.columns: true_ords.append("day")
    if "month" in df.columns: true_ords.append("month")

    # ord_3/4/5 -> nominal
    non_true = [c for c in candidate_ords if c not in true_ords]
    NOMINAL_COLS  = nominal + non_true
    ORDINAL_COLS  = true_ords
    CONTINUOUS_COLS = []

    # Explicit order maps; numeric ordinals get ascending order
    ORDINAL_ORDER_MAP = {
        "ord_1": ["Novice", "Contributor", "Expert", "Master", "Grandmaster"],
        "ord_2": ["Freezing", "Cold", "Warm", "Hot", "Boiling Hot", "Lava Hot"],
    }
    for col in ORDINAL_COLS:
        if df[col].dtype != object:
            vals = sorted(df[col].dropna().unique().tolist())
            ORDINAL_ORDER_MAP[col] = [str(v) for v in vals]

    used_cols = NOMINAL_COLS + ORDINAL_COLS + ["y"]
    data = df.dropna(subset=used_cols).reset_index(drop=True)
    y = data["y"].astype(int).values
    X = data.drop(columns=["y"]).reset_index(drop=True)

    pos = int((y==1).sum()); neg = int((y==0).sum())
    print(f"[Info] Final rows: {len(X)}  Positives: {pos}  Negatives: {neg}")
    print(f"[Info] NOMINAL: {NOMINAL_COLS}")
    print(f"[Info] ORDINAL: {ORDINAL_COLS}")
    print(f"[Info] CONTINUOUS: {CONTINUOUS_COLS}")
    return X, y, ORDINAL_ORDER_MAP, NOMINAL_COLS, ORDINAL_COLS, CONTINUOUS_COLS

# ---------------------------
# Helper: fit preprocessor once, transform splits
# ---------------------------
def fit_pre_and_transform(pre, X_tr, y_tr, X_val, X_te):
    pre_fit = clone(pre)
    t0 = time.time()
    pre_fit.fit(X_tr, y_tr)
    fit_s = time.time() - t0
    Z_tr = pre_fit.transform(X_tr)
    Z_val = pre_fit.transform(X_val)
    Z_te  = pre_fit.transform(X_te)
    return pre_fit, Z_tr, Z_val, Z_te, fit_s

def maybe_dense(M, model_name):
    if sp is not None and sp.issparse(M):
        if model_name in {"RF","KNN","LogReg"}:
            return M.toarray()
    return M

def xgb_predict_proba_safe(model, Z):
    """
    Version-safe proba prediction for xgboost.sklearn wrapper.
    Tries to use best_iteration if available; falls back gracefully.
    Returns positive-class probabilities (shape: [n,]).
    """
    try:
        if hasattr(model, "best_iteration") and model.best_iteration is not None:
            # Newer API
            proba = model.predict_proba(Z, iteration_range=(0, model.best_iteration + 1))
            return proba[:, 1]
    except TypeError:
        # iteration_range not supported in older builds
        pass
    # Fallback: plain predict_proba
    proba = model.predict_proba(Z)
    return proba[:, 1]

# ---------------------------
# Main runner
# ---------------------------
def run_all(csv_path: str):
    X, y, ORDINAL_ORDER_MAP, NOMINAL_COLS, ORDINAL_COLS, CONTINUOUS_COLS = load_catindat2(csv_path)

    # Splits
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=CONFIG["TEST_SIZE"], random_state=42, stratify=y)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=CONFIG["VAL_SIZE"], random_state=42, stratify=y_tr)

    preprocessors = build_preprocessors(NOMINAL_COLS, ORDINAL_COLS, CONTINUOUS_COLS, ORDINAL_ORDER_MAP)
    clf_dict = get_classifiers(y_tr)

    rows = []
    for p_idx, pre in enumerate(preprocessors, start=1):
        print(f"\n=== Fitting preprocessor for Pipeline P{p_idx} (once) ===")
        pre_fit, Z_tr, Z_val, Z_te, pre_secs = fit_pre_and_transform(pre, X_tr, y_tr, X_val, X_te)
        print(f"[P{p_idx}] preprocessor fit+transform done in {pre_secs:.1f}s")

        for name, clf in clf_dict.items():
            print(f"[P{p_idx} + {name}] training...")
            Ztr = maybe_dense(Z_tr, name)
            Zva = maybe_dense(Z_val, name)
            Zte = maybe_dense(Z_te, name)

            # Fit (XGB uses version-safe callback early stopping)
            t0 = time.time()
            if name == "XGB":
                clf_fit = clone(clf)
                # Use early_stopping_rounds for compatibility
                clf_fit.fit(
                    Ztr, y_tr,
                    eval_set=[(Zva, y_val)],
                    verbose=False
                )
            else:
                clf_fit = clone(clf).fit(Ztr, y_tr)
            fit_s = time.time() - t0

            # Scores on val/test
            if name == "XGB":
                prob_val = xgb_predict_proba_safe(clf_fit, Zva)
                prob_te  = xgb_predict_proba_safe(clf_fit, Zte)
            elif hasattr(clf_fit, "predict_proba"):
                prob_val = clf_fit.predict_proba(Zva)[:,1]
                prob_te  = clf_fit.predict_proba(Zte)[:,1]
            elif hasattr(clf_fit, "decision_function"):
                d_val = clf_fit.decision_function(Zva)
                d_te  = clf_fit.decision_function(Zte)
                prob_val = (d_val - d_val.min())/(d_val.max()-d_val.min()+1e-9)
                prob_te  = (d_te  - d_te.min() )/(d_te.max() - d_te.min() +1e-9)
            else:
                prob_val = clf_fit.predict(Zva)
                prob_te  = clf_fit.predict(Zte)

            # ---- Threshold tuning for Macro-F1 on validation (ALL models) ----
            t_best, f1_val = tune_threshold(prob_val, y_val, metric="macro_f1")
            y_pred_te = (prob_te >= t_best).astype(int)

            # Metrics
            acc  = accuracy_score(y_te, y_pred_te)
            f1m  = f1_score(y_te, y_pred_te, average="macro")
            f1p  = f1_score(y_te, y_pred_te, pos_label=1)
            roc  = roc_auc_score(y_te, prob_te)
            pr   = average_precision_score(y_te, prob_te)
            prec, rec, _, _ = precision_recall_fscore_support(y_te, y_pred_te, average=None, labels=[0,1])
            tn, fp, fn, tp   = confusion_matrix(y_te, y_pred_te).ravel()

            print(f"[P{p_idx} + {name}] Acc={acc:.4f} MacroF1={f1m:.4f} ROC={roc:.4f} PR+={pr:.4f} "
                  f"(t*={t_best:.3f}, valF1={f1_val:.4f}, train {fit_s:.1f}s)")

            rows.append({
                "Pipeline_Model": f"P{p_idx} - {name}",
                "Accuracy": round(acc,4),
                "MacroF1": round(f1m,4),
                "F1_Pos": round(f1p,4),
                "ROC_AUC": round(roc,4),
                "PR_AUC_Pos": round(pr,4),
                "Prec_Pos": round(prec[1],4),
                "Rec_Pos": round(rec[1],4),
                "TunedThreshold": round(t_best,3),
                "ValMacroF1@T": round(f1_val,4),
                "PreprocTime_s": round(pre_secs,1),
                "TrainTime_s": round(fit_s,1),
                "TP": tp, "FP": fp, "FN": fn, "TN": tn
            })

    out = pd.DataFrame(rows).sort_values(["MacroF1","PR_AUC_Pos","Accuracy"], ascending=False)
    print("\n=== Top models (by MacroF1, PR_AUC_Pos, Accuracy) ===")
    print(out.head(12).to_string(index=False))
    out.to_csv(CONFIG["RESULTS_CSV"], index=False)
    print(f"\nSaved: {CONFIG['RESULTS_CSV']}")
    return out

if __name__ == "__main__":
    run_all(CONFIG["CSV_PATH"])

[Info] Final rows: 300000  Positives: 91764  Negatives: 208236
[Info] NOMINAL: ['bin_0', 'bin_1', 'bin_2', 'bin_3', 'bin_4', 'nom_0', 'nom_1', 'nom_2', 'nom_3', 'nom_4', 'nom_5', 'nom_6', 'nom_7', 'nom_8', 'nom_9', 'ord_3', 'ord_4', 'ord_5']
[Info] ORDINAL: ['ord_0', 'ord_1', 'ord_2', 'day', 'month']
[Info] CONTINUOUS: []

=== Fitting preprocessor for Pipeline P1 (once) ===
[P1] preprocessor fit+transform done in 2.3s
[P1 + LogReg] training...
[P1 + LogReg] Acc=0.7025 MacroF1=0.6541 ROC=0.7319 PR+=0.5416 (t*=0.575, valF1=0.6599, train 45.1s)
[P1 + RF] training...
[P1 + RF] Acc=0.6988 MacroF1=0.6537 ROC=0.7287 PR+=0.5354 (t*=0.350, valF1=0.6576, train 349.3s)
[P1 + XGB] training...
[P1 + XGB] Acc=0.7233 MacroF1=0.6781 ROC=0.7604 PR+=0.5856 (t*=0.565, valF1=0.6834, train 774.5s)

=== Fitting preprocessor for Pipeline P2 (once) ===
[P2] preprocessor fit+transform done in 73.1s
[P2 + LogReg] training...
[P2 + LogReg] Acc=0.7008 MacroF1=0.6598 ROC=0.7384 PR+=0.5495 (t*=0.560, valF1=0.6632, 